# Exploring kernel functions

In this tutorial we will see all kernel functions currently available in `smudgy`.
We will deposit a particle dataset from a cosmological N-body simulation with different kernels and compare the resulting density maps.

First, let's load all necessary packages and modules.

In [1]:
import numpy as np
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import smudgy as sm
from smudgy import get_kernel_shapes_1D

In [2]:
try:
    plt.style.use('custom')
except:
    pass
%matplotlib inline
%config InlineBackend.figure_format='retina'

As discussed in the Kernels and Structures wiki page, `smudgy` supports `separable`, `isotropic` and `covariant` structures. Currently, only `tophat`, `tsc` and `gaussian` kernels can be used for `separable` structures, while all kernels can be used for the other two.

Let's define the structures and a dict with kernel names to analyze.

In [3]:
structures = ['separable', 'isotropic', 'covariant']

kernels_adaptive = {
    "Tophat": "tophat",
    "TSC": "tsc",
    "Gaussian": "gaussian",

    "Lucy": "lucy",
    "Cubic Spline": "cubic_spline",
    "Quintic Spline": "quintic_spline",

    "Wendland C2": "wendland_c2",
    "Wendland C4": "wendland_c4",
    "Wendland C6": "wendland_c6",
    }

batch_1 = {name: kernels_adaptive[name] for name in list(kernels_adaptive.keys())[0:3]}
batch_2 = {name: kernels_adaptive[name] for name in list(kernels_adaptive.keys())[3:6]}
batch_3 = {name: kernels_adaptive[name] for name in list(kernels_adaptive.keys())[6:9]}
batches = [batch_1, batch_2, batch_3]

# Pre-store colors for the kernels
colors_adaptive = [
    '#7F6330', '#DC511F', '#F7863E', '#F8A766', '#F8CB9B',
    '#2E984E', '#6ABC6F', '#97D393', '#BFE4BA',
][::-1]

## Kernel shape comparison

The following figure compares the one-dimensional profiles of all available kernels as a function of the normalized distance $q = \lVert\Delta\mathbfit{x}\rVert/h$ (see the {doc}`kernels and structures <../wiki/kernel_structures>` page for their definitions).

In [4]:
fig, ax = plt.subplots(1, len(batches), sharex=True, sharey=True, figsize=(6, 2))

c = 0
for i, batch in enumerate(batches):
    for kernel_name, kernel_key in batch.items():
        q, values = get_kernel_shapes_1D(kernel_key)
        ax[i].plot(q, values, label=kernel_name, lw=0.8, color=colors_adaptive[c])
        c += 1
    ax[i].legend(fontsize=5, loc='upper right')
    ax[i].set_xlabel(r"Normalized coordinate $q$")

ax[0].set(
       ylabel="Kernel value",
       ylim=(0, None)
       )

fig.savefig("kernel_shapes.png", dpi=500, bbox_inches="tight", transparent=False)
plt.close()

```{image} kernel_shapes.png
:width: 100%
:align: center
```

## Effect of the kernel on deposited maps

Different kernels are defined over different support domains. Some kernels exhibit sharper and more concentrated shapes, while others are more smoothly extended. Therefore, interpolation and deposition results almost always depend on the chosen kernel. We demonstrate this in the following workflow, where we deposit particles from a cosmological N-body simulation using different kernel functions.

As a reference, we first compute a grid using Nearest Grid Point assignment (NGP), which is often used in such applications, and then compare it to the maps obtained with the adaptive kernels.

We use the TNG50-4-Dark simulation from the publically available [TNG](https://www.tng-project.org/data/) runs as well as the `illustris_python` package to load `positions`, `masses` and the `boxsize` information.

In [5]:
import illustris_python as il

basePath = '/Users/mauro/work/projects/smudgy/data/tng50_4_dark/'
snapNum = 99
boxsize = 35
dm = il.snapshot.loadSubset(basePath, snapNum, "dm")

positions = dm['Coordinates'] / 1000
masses = np.ones(len(positions)) * 0.0186754872146347 * 1e10 # in Msun/h
positions = np.clip(positions, 0.01, boxsize-0.01)

Then we initialize a {py:class}`~smudgy.pointcloud.PointCloud` object, set some parameters to use during later computations and compute the smoothing lengths. For this tutorial we use the `isotropic` kernel structure. Since we are interested in the mass distribution, we deposit the particle weights (masses) themselves by passing `fields=None`, which returns the deposited mass in every cell.

In [6]:
pc = sm.PointCloud(
    positions=positions,
    weights=masses,
    boxsize=boxsize,
    verbose=False,
    )
num_neighbors = 32

kwargs = dict(
    fields=None,  # deposit the particle masses
    averaged=False,
    gridnums=64
)

pc.global_setup(num_neighbors=num_neighbors, structure='isotropic')
pc.compute_smoothing()

Next we deposit the particles and store the following grids:
+ A reference grid using the widely adopted NGP method (essentially a 3D histogram of the point cloud).
+ Grids using the adaptive kernels.

In [7]:
f_ref = pc.deposit(
    **kwargs,
    kernel_name='ngp',
    adaptive=False
)

maps_adaptive = []
for kernel_name, kernel_key in kernels_adaptive.items():

    f = pc.deposit(
        **kwargs,
        kernel_name=kernel_key,
        adaptive=True
    )
    maps_adaptive.append(f)

The figure below shows a visual comparison between the reference volume (left, NGP) and the adaptive kernels (right), projected along the first axis. All panels share the same logarithmic color scale.

In [8]:
ref_proj = f_ref.sum(axis=0)
ref_log = np.log10(ref_proj + 1)
vmin, vmax = np.percentile(ref_log, 1), np.percentile(ref_log, 99.99)


fig = plt.figure(figsize=(6, 3))
gs = gridspec.GridSpec(3, 4, width_ratios=[3, 1, 1, 1], wspace=0.01, hspace=0.01)

# Left Panel: Reference (NGP)
ax_ref = fig.add_subplot(gs[:, 0])
ax_ref.imshow(ref_log.T, origin='lower', cmap='bone', interpolation='none', vmin=vmin, vmax=vmax)
ax_ref.axis('off')
ax_ref.text(0.05, 0.95, "Reference (NGP)", color='white',
            transform=ax_ref.transAxes, va='top', fontsize=9, fontweight='bold')

for i, (name, _) in enumerate(kernels_adaptive.items()):
    row, col = divmod(i, 3)
    ax = fig.add_subplot(gs[row, col+1])

    kernel_proj = maps_adaptive[i].sum(axis=0)
    kernel_log = np.log10(kernel_proj + 1)

    ax.imshow(kernel_log.T, origin='lower', vmin=vmin, vmax=vmax, cmap='bone', interpolation='none')
    ax.text(0.05, 0.95, name, color='white', transform=ax.transAxes, va='top', fontsize=7)
    ax.axis('off')

fig.savefig("kernel_projection_comparison.png", dpi=500, bbox_inches="tight")
plt.close()

```{image} kernel_projection_comparison.png
:width: 100%
:align: center
```

On the same grid, the adaptive kernels resolve the filaments and haloes far more clearly than the NGP histogram, whose cells are dominated by discreteness noise. Differences between the individual adaptive kernels are subtle at this resolution.